# Stable Diffusion Deep Dive

Latent diffusion: instead of diffusing in pixel space (256x256x3 = 196K dims), compress to latent space (32x32x4 = 4K dims) with a pretrained VAE, then diffuse there. 48x fewer dimensions -- way cheaper to train and run.

We'll take apart every component of SD 1.5, touch each piece, and rebuild the pipeline manually.

**Time:** ~2-3 hrs. Needs ~6GB VRAM with FP16.

In [ ]:
# GPU: ~6 GB VRAM (FP16 model loading)
import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler
from transformers import CLIPTextModel, CLIPTokenizer
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import requests
from io import BytesIO
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to(device)
print(f"Pipeline loaded on {device}")

In [ ]:
# Inspect each component: architecture summary + param counts

def count_params(module: torch.nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())


def print_component_info(name: str, module: torch.nn.Module) -> None:
    params = count_params(module)
    print(f"\n{'='*60}")
    print(f"{name}: {params:,} parameters ({params * 2 / 1e9:.2f} GB in FP16)")
    print(f"{'='*60}")
    # Print top-level children only (avoid massive output)
    for child_name, child in module.named_children():
        child_params = count_params(child)
        print(f"  {child_name}: {child_params:,} params")


print_component_info("VAE", pipe.vae)
print_component_info("Text Encoder (CLIP)", pipe.text_encoder)
print_component_info("UNet", pipe.unet)

total = count_params(pipe.vae) + count_params(pipe.text_encoder) + count_params(pipe.unet)
print(f"\nTotal parameters: {total:,} ({total * 2 / 1e9:.2f} GB in FP16)")

# Input/output shapes
print("\n--- Expected I/O shapes ---")
print("VAE Encoder:  image (B, 3, 512, 512)  -> latent (B, 4, 64, 64)")
print("VAE Decoder:  latent (B, 4, 64, 64)   -> image  (B, 3, 512, 512)")
print("CLIP Encoder: token_ids (B, 77)        -> embeddings (B, 77, 768)")
print("UNet:         latent (B, 4, 64, 64) + t + text_emb (B, 77, 768) -> noise_pred (B, 4, 64, 64)")

In [ ]:
# VAE Encode: real image -> latent space

# Load a sample image
img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/300px-PNG_transparency_demonstration_1.png"
try:
    response = requests.get(img_url, timeout=10)
    source_image = Image.open(BytesIO(response.content)).convert("RGB")
except Exception:
    # Fallback: create a synthetic test image
    print("URL fetch failed, creating synthetic image")
    arr = np.zeros((512, 512, 3), dtype=np.uint8)
    # Gradient + shapes
    for i in range(512):
        arr[i, :, 0] = i // 2
        arr[:, i, 2] = i // 2
    arr[100:200, 100:200] = [255, 200, 0]  # yellow square
    arr[300:400, 300:400] = [0, 200, 255]   # cyan square
    source_image = Image.fromarray(arr)

source_image = source_image.resize((512, 512))

# Preprocess: to tensor, normalize to [-1, 1]
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
img_tensor = preprocess(source_image).unsqueeze(0).to(device, dtype=torch.float16)  # (1, 3, 512, 512)

# Encode
with torch.no_grad():
    latent_dist = pipe.vae.encode(img_tensor)
    latents = latent_dist.latent_dist.sample() * pipe.vae.config.scaling_factor

print(f"Image shape: {img_tensor.shape} -> Latent shape: {latents.shape}")
print(f"Compression ratio: {np.prod(img_tensor.shape[1:]) / np.prod(latents.shape[1:]):.1f}x")

# Visualize all 4 latent channels
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(source_image)
axes[0].set_title("Original")
axes[0].axis("off")

latent_np = latents[0].float().cpu().numpy()
for ch in range(4):
    axes[ch + 1].imshow(latent_np[ch], cmap="RdBu", vmin=-3, vmax=3)
    axes[ch + 1].set_title(f"Latent ch {ch}")
    axes[ch + 1].axis("off")

plt.suptitle("VAE Encoding: Image to 4-Channel Latent", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# VAE Decode: latent -> pixel space, compare with original

with torch.no_grad():
    decoded = pipe.vae.decode(latents / pipe.vae.config.scaling_factor).sample

# Convert back to displayable image
decoded_img = decoded[0].float().cpu().clamp(-1, 1)
decoded_img = (decoded_img + 1) / 2  # [-1,1] -> [0,1]
decoded_img = decoded_img.permute(1, 2, 0).numpy()

original_np = np.array(source_image).astype(np.float32) / 255.0

# Pixel difference
diff = np.abs(original_np - decoded_img)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(original_np)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(decoded_img)
axes[1].set_title("VAE Reconstructed")
axes[1].axis("off")

im = axes[2].imshow(diff.mean(axis=-1), cmap="hot", vmin=0, vmax=0.15)
axes[2].set_title("Pixel Difference (mean across channels)")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046)

mse = np.mean(diff ** 2)
psnr = -10 * np.log10(mse) if mse > 0 else float("inf")
plt.suptitle(f"VAE Reconstruction | MSE: {mse:.5f} | PSNR: {psnr:.1f} dB", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Text Encoding: CLIP tokenization + embedding

prompts = [
    "a painting of a cat sitting on a windowsill, impressionist style",
    "a photograph of an astronaut riding a horse on mars",
    "abstract colorful geometric shapes, minimalist",
]

fig, axes = plt.subplots(len(prompts), 1, figsize=(14, 3 * len(prompts)))

for idx, prompt in enumerate(prompts):
    # Tokenize
    tokens = pipe.tokenizer(
        prompt, padding="max_length", max_length=77, truncation=True, return_tensors="pt"
    )
    input_ids = tokens.input_ids.to(device)

    # Decode tokens back to text for inspection
    decoded_tokens = pipe.tokenizer.convert_ids_to_tokens(input_ids[0].cpu().tolist())
    # Find actual content tokens (not padding)
    eos_idx = decoded_tokens.index("<|endoftext|>") if "<|endoftext|>" in decoded_tokens else 77
    content_tokens = decoded_tokens[:eos_idx + 1]

    print(f"\nPrompt: '{prompt}'")
    print(f"Token IDs (first {len(content_tokens)}): {input_ids[0, :len(content_tokens)].cpu().tolist()}")
    print(f"Tokens: {content_tokens}")

    # Encode
    with torch.no_grad():
        text_emb = pipe.text_encoder(input_ids)[0]  # (1, 77, 768)
    print(f"Embedding shape: {text_emb.shape}")

    # Per-token embedding norm
    norms = text_emb[0].float().norm(dim=-1).cpu().numpy()  # (77,)
    axes[idx].bar(range(77), norms, color="steelblue", alpha=0.7)
    axes[idx].set_title(f"'{prompt[:60]}...'" if len(prompt) > 60 else f"'{prompt}'")
    axes[idx].set_xlabel("Token position")
    axes[idx].set_ylabel("L2 norm")
    axes[idx].axvline(x=eos_idx, color="red", linestyle="--", alpha=0.5, label="EOS")
    axes[idx].legend()

plt.suptitle("CLIP Text Embeddings: Per-Token Norms", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Manual pipeline: step-by-step Stable Diffusion
# GPU: ~6 GB VRAM

# 1. Encode text
prompt = "a painting of a cat sitting on a windowsill, impressionist style"
text_input = pipe.tokenizer(
    prompt, padding="max_length", max_length=77, return_tensors="pt"
).to(device)
with torch.no_grad():
    text_embeddings = pipe.text_encoder(text_input.input_ids)[0]  # (1, 77, 768)

# Unconditional embeddings for classifier-free guidance
uncond_input = pipe.tokenizer(
    "", padding="max_length", max_length=77, return_tensors="pt"
).to(device)
with torch.no_grad():
    uncond_embeddings = pipe.text_encoder(uncond_input.input_ids)[0]

# Concat: [unconditional, conditional] for batched CFG
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])  # (2, 77, 768)
print(f"Text embeddings shape: {text_embeddings.shape}")

# 2. Initialize noise
generator = torch.Generator(device=device).manual_seed(42)
latents = torch.randn(
    (1, 4, 64, 64), generator=generator, device=device, dtype=torch.float16
)

# 3. Setup DDIM scheduler
scheduler = DDIMScheduler.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", subfolder="scheduler"
)
num_inference_steps = 50
scheduler.set_timesteps(num_inference_steps)
latents = latents * scheduler.init_noise_sigma

print(f"Scheduler timesteps ({len(scheduler.timesteps)}): {scheduler.timesteps[:10].tolist()} ...")

# 4. Denoising loop with classifier-free guidance
guidance_scale = 7.5
intermediates = []

for i, t in enumerate(tqdm(scheduler.timesteps, desc="Denoising")):
    # Duplicate latents for CFG (unconditional + conditional)
    latent_model_input = torch.cat([latents] * 2)
    latent_model_input = scheduler.scale_model_input(latent_model_input, t)

    # UNet forward pass
    with torch.no_grad():
        noise_pred = pipe.unet(
            latent_model_input, t, encoder_hidden_states=text_embeddings
        ).sample

    # CFG: noise_pred = uncond + guidance_scale * (cond - uncond)
    noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

    # Scheduler step
    latents = scheduler.step(noise_pred, t, latents).prev_sample

    # Save intermediates every 10 steps
    if i % 10 == 0:
        intermediates.append((i, t.item(), latents.clone()))

# 5. Decode final latent
with torch.no_grad():
    image = pipe.vae.decode(latents / pipe.vae.config.scaling_factor).sample

# Convert to displayable
image_np = image[0].float().cpu().clamp(-1, 1)
image_np = ((image_np + 1) / 2).permute(1, 2, 0).numpy()

plt.figure(figsize=(6, 6))
plt.imshow(image_np)
plt.title(f"'{prompt}'\n(50 DDIM steps, CFG=7.5)")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Display intermediate denoising steps

n_intermediates = len(intermediates)
fig, axes = plt.subplots(1, n_intermediates, figsize=(4 * n_intermediates, 4))

for idx, (step_i, t_val, lat) in enumerate(intermediates):
    with torch.no_grad():
        decoded = pipe.vae.decode(lat / pipe.vae.config.scaling_factor).sample
    img = decoded[0].float().cpu().clamp(-1, 1)
    img = ((img + 1) / 2).permute(1, 2, 0).numpy()
    axes[idx].imshow(img)
    axes[idx].set_title(f"Step {step_i}\n(t={t_val})")
    axes[idx].axis("off")

plt.suptitle("Denoising Progression (DDIM)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Latent space interpolation via SLERP
# GPU: ~6 GB VRAM


def slerp(t: float, v0: torch.Tensor, v1: torch.Tensor) -> torch.Tensor:
    """Spherical linear interpolation between two tensors."""
    v0_flat = v0.flatten().float()
    v1_flat = v1.flatten().float()
    v0_norm = v0_flat / v0_flat.norm()
    v1_norm = v1_flat / v1_flat.norm()
    omega = torch.acos(torch.clamp(torch.dot(v0_norm, v1_norm), -1.0, 1.0))
    if omega.abs() < 1e-6:
        return ((1.0 - t) * v0_flat + t * v1_flat).reshape(v0.shape).to(v0.dtype)
    sin_omega = torch.sin(omega)
    result = (torch.sin((1.0 - t) * omega) / sin_omega) * v0_flat + \
             (torch.sin(t * omega) / sin_omega) * v1_flat
    return result.reshape(v0.shape).to(v0.dtype)


# Generate two different noise tensors
gen_a = torch.Generator(device=device).manual_seed(42)
noise_a = torch.randn((1, 4, 64, 64), generator=gen_a, device=device, dtype=torch.float16)

gen_b = torch.Generator(device=device).manual_seed(123)
noise_b = torch.randn((1, 4, 64, 64), generator=gen_b, device=device, dtype=torch.float16)

# Create interpolation points
n_interp = 9
interp_values = np.linspace(0.0, 1.0, n_interp)
interpolated_noises = [slerp(t, noise_a, noise_b) for t in interp_values]

# Denoise each interpolated noise
interp_prompt = "a portrait of a woman, oil painting, renaissance style"
text_input_interp = pipe.tokenizer(
    interp_prompt, padding="max_length", max_length=77, return_tensors="pt"
).to(device)
with torch.no_grad():
    text_emb_interp = pipe.text_encoder(text_input_interp.input_ids)[0]
uncond_input_interp = pipe.tokenizer(
    "", padding="max_length", max_length=77, return_tensors="pt"
).to(device)
with torch.no_grad():
    uncond_emb_interp = pipe.text_encoder(uncond_input_interp.input_ids)[0]
text_emb_combined = torch.cat([uncond_emb_interp, text_emb_interp])

interp_scheduler = DDIMScheduler.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", subfolder="scheduler"
)
interp_steps = 30  # fewer steps for speed
interp_scheduler.set_timesteps(interp_steps)

interp_images = []
for i, noise in enumerate(tqdm(interpolated_noises, desc="Interpolating")):
    latents_i = noise * interp_scheduler.init_noise_sigma
    for t in interp_scheduler.timesteps:
        lat_input = torch.cat([latents_i] * 2)
        lat_input = interp_scheduler.scale_model_input(lat_input, t)
        with torch.no_grad():
            noise_pred = pipe.unet(lat_input, t, encoder_hidden_states=text_emb_combined).sample
        uncond_pred, text_pred = noise_pred.chunk(2)
        noise_pred = uncond_pred + 7.5 * (text_pred - uncond_pred)
        latents_i = interp_scheduler.step(noise_pred, t, latents_i).prev_sample

    with torch.no_grad():
        decoded = pipe.vae.decode(latents_i / pipe.vae.config.scaling_factor).sample
    img = decoded[0].float().cpu().clamp(-1, 1)
    img = ((img + 1) / 2).permute(1, 2, 0).numpy()
    interp_images.append(img)

# Display interpolation strip
fig, axes = plt.subplots(1, n_interp, figsize=(3 * n_interp, 3))
for i, img in enumerate(interp_images):
    axes[i].imshow(img)
    axes[i].set_title(f"t={interp_values[i]:.2f}")
    axes[i].axis("off")

plt.suptitle(f"SLERP Interpolation: '{interp_prompt}'", fontsize=13)
plt.tight_layout()
plt.show()

## Checkpoint

Draw the SD pipeline from memory:

```
Text -> CLIP -> embeddings -> cross-attention in UNet
                                    |
Noise -> UNet(noise, t, text_emb) -> predicted_noise -> scheduler_step -> repeat -> latent -> VAE_decode -> image
```

The key insight: diffusion happens in a 4-channel latent space, not pixel space. The VAE provides a learned compression that preserves perceptual quality while reducing dimensionality by ~48x.

**Further reading:**
- Latent Diffusion Models: [arXiv:2112.10752](https://arxiv.org/abs/2112.10752)